## Cross-Validation for Regularization

Common regularized regression objectives include:

* **Ridge Regression ($L_2$ Regularization):**

$$\mathcal{L}_{\text{Ridge}}(\beta) = \frac{1}{N} \sum_{i=1}^{N} (y_i - x_i^T \beta)^2 + \alpha \sum_{j=1}^{d} \beta_j^2$$


* **Lasso Regression ($L_1$ Regularization):**

$$\mathcal{L}_{\text{Lasso}}(\beta) = \frac{1}{N} \sum_{i=1}^{N} (y_i - x_i^T \beta)^2 + \alpha \sum_{j=1}^{d} \vert{}\beta_j\vert{}$$



Here, $\alpha \ge 0$ (often denoted as $\lambda$) is a **hyperparameter** governing the strength of the regularization:

* **$\alpha \to 0$:** Unregularized model (High variance, risk of overfitting).
* **$\alpha \to \infty$:** Strong penalty (High bias, parameters shrink to zero, risk of underfitting).

Because $\alpha$ is a hyperparameter (not learned during gradient descent or closed-form optimization), we must tune it using **Cross-Validation (CV)**.

---

## Preventing Data Leakage in Manual Cross-Validation

Regularized algorithms (especially Ridge and Lasso) are scale-sensitive. Features must be standardized (e.g., zero mean, unit variance) so that penalties are applied uniformly across all dimensions.

> **Critical Rule:** Standardization parameters ($\mu$ and $\sigma$) must be computed **only** on the training fold and then applied to the validation fold. Fitting a scaler on the entire dataset prior to splitting introduces **data leakage**, resulting in overly optimistic validation performance.

---

## The $K$-Fold CV Tuning Algorithm

To select the optimal $\alpha^*$ across a candidate grid $\mathcal{A} = \{\alpha_1, \alpha_2, \dots, \alpha_M\}$ using $K$-Fold Cross-Validation:

1. Divide the training dataset $D$ randomly into $K$ equal-sized folds: $D_1, D_2, \dots, D_K$.
2. For each hyperparameter value $\alpha_m \in \mathcal{A}$:
* For each fold $k \in \{1, 2, \dots, K\}$:
1. Treat $D_k$ as the **validation set** and $D_{\setminus k} = D \setminus D_k$ as the **training fold**.
2. Fit the feature scaler on $D_{\setminus k}$ to compute mean $\mu_{\setminus k}$ and standard deviation $\sigma_{\setminus k}$.
3. Transform $D_{\setminus k}$ and $D_k$ using $\mu_{\setminus k}$ and $\sigma_{\setminus k}$.
4. Train the regularized model on the scaled $D_{\setminus k}$ using parameter $\alpha_m$.
5. Evaluate error (e.g., Mean Squared Error) on the scaled validation set $D_k$: $\text{MSE}_{k}(\alpha_m)$.




3. Compute the mean cross-validation error for each candidate parameter $\alpha_m$:

$$\text{CV_Error}(\alpha_m) = \frac{1}{K} \sum_{k=1}^{K} \text{MSE}_{k}(\alpha_m)$$


4. Select the optimal parameter:

$$\alpha^* = \arg\min_{\alpha_m \in \mathcal{A}} \text{CV\_Error}(\alpha_m)$$


5. **Final Step (Refitting):** Fit the scaler on the **entire dataset $D$**, transform $D$, and train the final model on all scaled data using $\alpha^*$.

---

## 4. Python Implementation (Explicit Loop, No Pipelines)

The following implementation uses `scikit-learn` strictly for model primitives (`Ridge`, `StandardScaler`, `KFold`, `mean_squared_error`), executing fold creation, scaling, and evaluation explicitly.

In [1]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------------------------
# 1. Setup Synthetic Data & Hyperparameter Search Space
# -------------------------------------------------------------------
X, y = make_regression(n_samples=300, n_features=20, noise=10.0, random_state=42)

# Candidate alphas spanning logarithmic scale from 10^-3 to 10^3
alpha_grid = np.logspace(-3, 3, 50)
K = 5  # 5-fold cross-validation

# Store validation MSE: rows = alphas, columns = folds
cv_errors = np.zeros((len(alpha_grid), K))

# -------------------------------------------------------------------
# 2. Manual K-Fold Cross-Validation Loop
# -------------------------------------------------------------------
kf = KFold(n_splits=K, shuffle=True, random_state=42)

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    # Split raw data for the current fold
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y[train_idx], y[val_idx]

    # Preprocessing: Fit scaler ONLY on the fold training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fold)

    # Apply the training fold's scaling parameters to the validation fold
    X_val_scaled = scaler.transform(X_val_fold)

    # Evaluate every alpha candidate on this fold
    for alpha_idx, alpha in enumerate(alpha_grid):
        # Instantiate and fit model on the scaled fold training data
        model = Ridge(alpha=alpha)
        model.fit(X_train_scaled, y_train_fold)

        # Predict on the scaled validation data
        y_val_pred = model.predict(X_val_scaled)

        # Calculate and record validation error
        cv_errors[alpha_idx, fold_idx] = mean_squared_error(y_val_fold, y_val_pred)

# -------------------------------------------------------------------
# 3. Model Selection
# -------------------------------------------------------------------
# Average validation MSE across all K folds for each candidate alpha
mean_cv_errors = cv_errors.mean(axis=1)

# Find the index of the alpha yielding the minimum mean CV error
best_alpha_idx = np.argmin(mean_cv_errors)
best_alpha = alpha_grid[best_alpha_idx]

print(f"Optimal Alpha (λ): {best_alpha:.4f}")
print(f"Lowest Mean CV MSE: {mean_cv_errors[best_alpha_idx]:.4f}")

# -------------------------------------------------------------------
# 4. Final Model Retraining (Full Dataset)
# -------------------------------------------------------------------
# Fit scaler on full training set
full_scaler = StandardScaler()
X_scaled_full = full_scaler.fit_transform(X)

# Train production model on full scaled dataset using best alpha
final_model = Ridge(alpha=best_alpha)
final_model.fit(X_scaled_full, y)

Optimal Alpha (λ): 0.0010
Lowest Mean CV MSE: 106.1210


Ridge(alpha=np.float64(0.001))

---

1. **Hyperparameter Selection vs. Model Parameters:** Regularization parameters ($\alpha$ or $\lambda$) cannot be learned directly via standard objective minimization because doing so would always drive $\alpha \to 0$ to minimize training error.
2. **Proper Data Splitting:** Preprocessing transforms (scaling, imputation, dimensionality reduction) must be calculated inside each cross-validation split independently.
3. **Refitting is Required:** Once the optimal parameter $\alpha^*$ is determined, retrain the final model on the full training dataset using $\alpha^*$ to maximize data utilization before test-set deployment.